# VisionBridge - Real Video Base Model Validation v2

Run every cell from top to bottom. Upload one real ISL sentence video, enter its English ground truth, extract real MediaPipe pose/face keypoints, load the committed base model, and print the prediction report.

Validation only. No training and no model modification.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil
REPO = Path('/content/VisionBridge')
if not (REPO / 'README.md').exists():
    subprocess.run(['git','clone','https://github.com/BharathWaj-K-R/VisionBridge.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'checkout','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'], check=True)
sys.path.insert(0, str(REPO / 'backend'))
print('Repository:', REPO)
print('HEAD:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'], text=True).strip())

In [ ]:
from google.colab import files
print('Upload ONE real ISL sentence video: mp4, mov, avi, or mkv')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No video uploaded.')
VIDEO_SRC = Path('/content') / next(iter(uploaded))
if VIDEO_SRC.suffix.lower() not in {'.mp4','.mov','.avi','.mkv'}:
    raise ValueError(f'Unsupported format: {VIDEO_SRC.suffix}')
GROUND_TRUTH = input('Ground truth English sentence (optional): ').strip()
print('Selected:', VIDEO_SRC.name)
print('Ground truth:', GROUND_TRUTH or '(not supplied)')

In [ ]:
from pathlib import Path
import os, subprocess, shutil
UV = shutil.which('uv') or '/usr/local/bin/uv'
if not Path(UV).exists():
    subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
    UV = '/root/.local/bin/uv'
MP_ENV = Path('/content/visionbridge_mp312')
MP_PYTHON = MP_ENV / 'bin' / 'python'
MPL_CONFIG = Path('/content/visionbridge_mplconfig')
MPL_CONFIG.mkdir(parents=True, exist_ok=True)
if not MP_ENV.exists():
    subprocess.run([UV,'python','install','3.12'], check=True)
    subprocess.run([UV,'venv','--python','3.12',str(MP_ENV)], check=True)
env = os.environ.copy()
env['MPLBACKEND'] = 'Agg'
env['MPLCONFIGDIR'] = str(MPL_CONFIG)
probe = subprocess.run([str(MP_PYTHON), '-c', 'import mediapipe; from mediapipe.python.solutions import holistic; print(mediapipe.__version__)'], text=True, capture_output=True, env=env)
if probe.returncode != 0 or probe.stdout.strip() != '0.10.21':
    subprocess.run([UV,'pip','install','--python',str(MP_PYTHON),'mediapipe==0.10.21','numpy==1.26.4','opencv-python-headless','pandas','matplotlib'], check=True, env=env)
probe = subprocess.run([str(MP_PYTHON), '-c', 'import mediapipe; from mediapipe.python.solutions import holistic; print(mediapipe.__version__)'], text=True, capture_output=True, env=env)
print(probe.stdout)
if probe.returncode != 0:
    print(probe.stderr)
    raise RuntimeError('MediaPipe environment validation failed.')
print('MEDIAPIPE ENV: PASS')

In [ ]:
import csv, numpy as np
CHECK_DIR = REPO / 'data' / 'model_check'
VIDEOS = CHECK_DIR / 'videos'
PROCESSED = CHECK_DIR / 'processed'
VIDEOS.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
UID = VIDEO_SRC.stem
video_copy = VIDEOS / VIDEO_SRC.name
shutil.copy2(VIDEO_SRC, video_copy)
LABELS = CHECK_DIR / 'validation_labels.csv'
with LABELS.open('w', newline='', encoding='utf-8') as f:
    w = csv.writer(f); w.writerow(['uid','text']); w.writerow([UID, GROUND_TRUTH or 'validation sample'])
cmd = [str(MP_PYTHON), str(REPO/'backend'/'scripts'/'extract_keypoints.py'), '--videos_dir', str(VIDEOS), '--labels_csv', str(LABELS), '--out_dir', str(PROCESSED)]
result = subprocess.run(cmd, cwd=str(REPO), env=env, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Real MediaPipe extraction failed.')
pose = np.load(PROCESSED/'pose'/f'{UID}.npy')
face = np.load(PROCESSED/'face'/f'{UID}.npy')
print('Pose shape:', pose.shape)
print('Face shape:', face.shape)
assert pose.ndim == 2 and pose.shape[1] == 132
assert face.ndim == 2 and face.shape[1] == 1404
assert pose.shape[0] == face.shape[0]
print('REAL KEYPOINT EXTRACTION: PASS')

In [ ]:
import torch
from app.models.base_model import load_frozen_base_model
from app.training.isltranslate import SimpleCharTokenizer, _downsample_to_max_length
WEIGHTS = REPO/'backend/app/models/weights/base_model.pt'
VOCAB = REPO/'backend/app/models/weights/base_model.vocab.json'
if not WEIGHTS.exists(): raise FileNotFoundError(WEIGHTS)
if not VOCAB.exists(): raise FileNotFoundError(VOCAB)
tokenizer = SimpleCharTokenizer.load(VOCAB)
model = load_frozen_base_model(str(WEIGHTS), vocab_size=tokenizer.vocab_size)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device).eval()
pose_t, face_t = _downsample_to_max_length(torch.from_numpy(pose).float(), torch.from_numpy(face).float(), UID)
pose_t = pose_t.unsqueeze(0).to(device)
face_t = face_t.unsqueeze(0).to(device)
lengths = torch.tensor([pose_t.shape[1]], dtype=torch.long, device=device)
with torch.no_grad():
    logits = model(pose_t, face_t, lengths=lengths)
print('Device:', device)
print('Logits shape:', tuple(logits.shape))
print('Logits finite:', bool(torch.isfinite(logits).all()))

In [ ]:
logit = logits[0, :int(lengths[0].item())]
probs = torch.softmax(logit, dim=-1)
frame_ids = probs.argmax(dim=-1)
frame_conf = probs.max(dim=-1).values
collapsed = []
previous = None
for token_id in frame_ids.tolist():
    if token_id != previous: collapsed.append(token_id)
    previous = token_id
decoded_ids = [i for i in collapsed if i != 0]
prediction = ''.join(tokenizer.id_to_token[i] for i in decoded_ids if 0 <= i < len(tokenizer.id_to_token) and tokenizer.id_to_token[i] != tokenizer.blank_token).strip() or '(no sign detected)'
blank_ratio = float((frame_ids == 0).float().mean().item())
non_blank = int((frame_ids != 0).sum().item())
confidence = float(frame_conf.mean().item())
def levenshtein(a, b):
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0]*len(b)
        for j, cb in enumerate(b, 1): cur[j] = min(prev[j]+1, cur[j-1]+1, prev[j-1]+(ca != cb))
        prev = cur
    return prev[-1]
cer = None if not GROUND_TRUTH else levenshtein(prediction.lower(), GROUND_TRUTH.lower()) / max(1, len(GROUND_TRUTH))
print('\n' + '='*70)
print('VISIONBRIDGE REAL-VIDEO MODEL CHECK')
print('='*70)
print('VIDEO:          ', VIDEO_SRC.name)
print('DEVICE:         ', device)
print('FRAMES:         ', frame_ids.numel())
print('GROUND TRUTH:   ', GROUND_TRUTH or '(not supplied)')
print('PREDICTED:      ', prediction)
print('CONFIDENCE:     ', f'{confidence:.3f}')
print('BLANK RATIO:    ', f'{blank_ratio:.4f}')
print('NON-BLANK:      ', non_blank)
print('UNIQUE TOKENS:  ', len(set(decoded_ids)))
print('LOGITS FINITE:  ', bool(torch.isfinite(logits).all()))
if cer is not None: print('CER:             ', f'{cer:.4f}')
print('='*70)
print('RESULT:', 'MODEL PRODUCED ONLY CTC BLANKS.' if blank_ratio == 1.0 else 'NON-BLANK OUTPUT PRODUCED.')

In [ ]:
from IPython.display import Video, display
display(Video(str(video_copy), embed=True))